In [1]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Enable a GPU runtime.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

Wed Nov 18 17:38:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers accelerate bitsandbytes

In [3]:
import os
import gc
import json
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 128
CONTEXT_LIMIT = 2048

OUTPUT_DIR = "/content/task3_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

prompts = [
    "Explain machine learning in simple terms.",
    "What is the difference between AI and deep learning?",
    "Write a Python function to check whether a number is prime.",
    "Explain REST APIs with a practical example.",
    "What are the advantages of database normalization?",
    "Explain overfitting and how to prevent it.",
    "What is cloud computing? Describe its main service models.",
    "Explain the working of a binary search algorithm.",
    "What are the major challenges in deploying AI systems?",
    "Write a short explanation of cybersecurity best practices."
]

print("Model:", MODEL_NAME)
print("Number of prompts:", len(prompts))
print("GPU:", torch.cuda.get_device_name(0))

Model: Qwen/Qwen2.5-1.5B-Instruct
Number of prompts: 10
GPU: Tesla T4


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def benchmark_model(model, configuration_name):

    results = []

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    total_tokens = 0
    total_time = 0

    for i, prompt in enumerate(prompts, start=1):

        messages = [
            {"role": "user", "content": prompt}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        )

        inputs = inputs.to("cuda")
        input_length = inputs["input_ids"].shape[1]

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        generated_ids = output[0][input_length:]

        generated_tokens = generated_ids.shape[0]

        answer = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        total_tokens += generated_tokens
        total_time += elapsed

        results.append({
            "prompt_id": i,
            "prompt": prompt,
            "answer": answer,
            "input_tokens": input_length,
            "generated_tokens": generated_tokens,
            "generation_time_seconds": round(elapsed, 4),
            "generation_tokens_per_second": round(
                generated_tokens / elapsed, 2
            )
        })

        print(f"\nPrompt {i}/10")
        print("Question:", prompt)
        print("Answer:", answer)
        print("Generated tokens:", generated_tokens)
        print("Time:", round(elapsed, 2), "seconds")

    peak_allocated = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3

    summary = {
        "model": MODEL_NAME,
        "configuration": configuration_name,
        "number_of_prompts": len(results),
        "total_generated_tokens": total_tokens,
        "total_generation_time_seconds": round(total_time, 3),
        "average_generation_tokens_per_second": round(
            total_tokens / total_time, 2
        ),
        "peak_gpu_memory_allocated_gb": round(peak_allocated, 3),
        "peak_gpu_memory_reserved_gb": round(peak_reserved, 3),
        "results": results
    }

    return summary

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
import time

gc.collect()
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

torch.cuda.synchronize()

fp16_load_time = time.perf_counter() - start

fp16_load_memory = (
    torch.cuda.memory_allocated() / 1024**3
)

print("FP16 model loaded.")
print("Loading time:", round(fp16_load_time, 2), "seconds")
print("GPU memory allocated:", round(fp16_load_memory, 2), "GB")

fp16_summary = benchmark_model(model_fp16, "FP16")

fp16_summary["model_loading_time_seconds"] = round(
    fp16_load_time, 2
)

fp16_summary["model_loading_memory_gb"] = round(
    fp16_load_memory, 3
)

with open(
    os.path.join(OUTPUT_DIR, "fp16_results.json"), "w"
) as f:
    json.dump(fp16_summary, f, indent=4)

print("\nFP16 benchmark saved.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

FP16 model loaded.
Loading time: 82.46 seconds
GPU memory allocated: 2.88 GB

Prompt 1/10
Question: Explain machine learning in simple terms.
Answer: Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms and statistical models to analyze data and make predictions or decisions based on patterns found within the data.

In simpler terms, imagine you have a large dataset containing information about different types of fruits (e.g., apples, bananas, oranges). Machine learning can help us create a model that learns how these fruits behave and classify new fruits into their correct categories based on this learned behavior. This way, we don't need to manually label each fruit; instead, the computer uses its "learning" ability to recognize patterns and categor
Generated tokens: 128
Time: 7.05 seconds

Prompt 2/10
Question: What is the difference between AI and deep learn

In [6]:
del model_fp16

gc.collect()
torch.cuda.empty_cache()

print("GPU memory after FP16 cleanup:")
!nvidia-smi

GPU memory after FP16 cleanup:
Wed Nov 18 17:44:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             32W /   70W |    3077MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------

In [7]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

model_q4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

torch.cuda.synchronize()

q4_load_time = time.perf_counter() - start

q4_load_memory = (
    torch.cuda.memory_allocated() / 1024**3
)

print("4-bit model loaded.")
print("Loading time:", round(q4_load_time, 2), "seconds")
print("GPU memory allocated:", round(q4_load_memory, 2), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit model loaded.
Loading time: 14.43 seconds
GPU memory allocated: 1.08 GB


In [8]:
q4_summary = benchmark_model(model_q4, "4-bit NF4")

q4_summary["model_loading_time_seconds"] = round(
    q4_load_time, 2
)

q4_summary["model_loading_memory_gb"] = round(
    q4_load_memory, 3
)

with open(
    os.path.join(OUTPUT_DIR, "q4_results.json"), "w"
) as f:
    json.dump(q4_summary, f, indent=4)

print("\n4-bit benchmark saved.")


Prompt 1/10
Question: Explain machine learning in simple terms.
Answer: Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance on specific tasks over time without being explicitly programmed. It involves using algorithms and statistical models to analyze large datasets and identify patterns or trends that can be used for prediction or decision-making.

In simpler terms, imagine you have a group of people who each have different heights. If someone wants to predict the height of another person based on this information, they would need to know all the individual heights. However, if we use machine learning, we can train a model to recognize patterns in the data (in this case, the relationship between two variables) and
Generated tokens: 128
Time: 10.62 seconds

Prompt 2/10
Question: What is the difference between AI and deep learning?
Answer: Artificial Intelligence (AI) refers to the simulation of human intelligence

In [9]:
comparison = pd.DataFrame([
    {
        "Configuration": "FP16",
        "Loading time (s)": fp16_summary["model_loading_time_seconds"],
        "Generation speed (tokens/s)": fp16_summary[
            "average_generation_tokens_per_second"
        ],
        "Peak allocated GPU memory (GB)": fp16_summary[
            "peak_gpu_memory_allocated_gb"
        ],
        "Peak reserved GPU memory (GB)": fp16_summary[
            "peak_gpu_memory_reserved_gb"
        ]
    },
    {
        "Configuration": "4-bit NF4",
        "Loading time (s)": q4_summary["model_loading_time_seconds"],
        "Generation speed (tokens/s)": q4_summary[
            "average_generation_tokens_per_second"
        ],
        "Peak allocated GPU memory (GB)": q4_summary[
            "peak_gpu_memory_allocated_gb"
        ],
        "Peak reserved GPU memory (GB)": q4_summary[
            "peak_gpu_memory_reserved_gb"
        ]
    }
])

display(comparison)

comparison.to_csv(
    os.path.join(OUTPUT_DIR, "benchmark_comparison.csv"),
    index=False
)

print("Comparison saved.")

,Configuration,Loading time (s),Generation speed (tokens/s),Peak allocated GPU memory (GB),Peak reserved GPU memory (GB)
0,FP16,82.46,26.51,2.890,2.934
1,4-bit NF4,14.43,12.83,1.092,2.918


Comparison saved.


In [10]:
quality_rows = []

for fp, q4 in zip(
    fp16_summary["results"],
    q4_summary["results"]
):
    quality_rows.append({
        "Prompt ID": fp["prompt_id"],
        "Prompt": fp["prompt"],
        "FP16 Answer": fp["answer"],
        "4-bit Answer": q4["answer"],
        "Quality Assessment": "",
        "Observations": ""
    })

quality_df = pd.DataFrame(quality_rows)

quality_df.to_csv(
    os.path.join(OUTPUT_DIR, "quality_comparison.csv"),
    index=False
)

print("Quality comparison template saved.")
display(quality_df[["Prompt ID", "Prompt", "FP16 Answer", "4-bit Answer"]])

Quality comparison template saved.


,Prompt ID,Prompt,FP16 Answer,4-bit Answer
0,1,Explain machine learning in simple terms.,Machine learning is a type of artificial intel...,Machine learning is a type of artificial intel...
1,2,What is the difference between AI and deep lea...,Artificial Intelligence (AI) refers to the sim...,Artificial Intelligence (AI) refers to the sim...
2,3,Write a Python function to check whether a num...,Certainly! Here's a simple Python function tha...,Certainly! Here's a Python function that check...
3,4,Explain REST APIs with a practical example.,REST (Representational State Transfer) is an a...,RESTful (Representational State Transfer) is a...
4,5,What are the advantages of database normalizat...,Database normalization is a process that organ...,Database normalization is a process that organ...
5,6,Explain overfitting and how to prevent it.,Overfitting is a common problem in machine lea...,Overfitting is a common problem in machine lea...
6,7,What is cloud computing? Describe its main ser...,Cloud computing refers to the delivery of comp...,Cloud computing refers to the delivery of comp...
7,8,Explain the working of a binary search algorithm.,Binary Search is an efficient searching algori...,Binary search is an efficient searching algori...
8,9,What are the major challenges in deploying AI ...,Deploying AI systems can present several signi...,Deploying AI systems can be challenging due to...
9,10,Write a short explanation of cybersecurity bes...,Cybersecurity best practices refer to the set ...,Cybersecurity best practices refer to the set ...


In [11]:
from google.colab import drive

drive.mount("/content/drive")

import shutil

DRIVE_DIR = (
    "/content/drive/MyDrive/Tndrr_Task3_Inference"
)

os.makedirs(DRIVE_DIR, exist_ok=True)

for filename in os.listdir(OUTPUT_DIR):
    source = os.path.join(OUTPUT_DIR, filename)
    destination = os.path.join(DRIVE_DIR, filename)

    shutil.copy2(source, destination)

print("Task 3 results saved to Google Drive:")
for filename in sorted(os.listdir(DRIVE_DIR)):
    print("-", filename)

Mounted at /content/drive
Task 3 results saved to Google Drive:
- benchmark_comparison.csv
- fp16_results.json
- q4_results.json
- quality_comparison.csv


In [12]:
import json
import os

# Calculate theoretical KV cache for this model
# Qwen2.5-1.5B: 28 layers, 2 KV heads, head dimension 128
layers = 28
kv_heads = 2
head_dim = 128
bytes_per_value = 2
users = 20
tokens_per_user = 2048

kv_cache_bytes = (
    2 * layers * kv_heads * head_dim
    * bytes_per_value * users * tokens_per_user
)

kv_cache_gib = kv_cache_bytes / (1024 ** 3)

readme = f"""
# Task 3: Local LLM Inference and Quantization

## Objective
Compare FP16 and 4-bit quantized inference using the same
model, prompts, generation settings and GPU environment.

## Model and environment
- Model: {MODEL_NAME}
- Runtime: Hugging Face Transformers
- Quantization: bitsandbytes NF4, double quantization enabled
- GPU: {torch.cuda.get_device_name(0)}
- Evaluation prompts: 10
- Maximum generated tokens: {MAX_NEW_TOKENS}
- Sampling: Disabled (greedy decoding)

## Benchmark results

| Metric | FP16 | 4-bit NF4 |
|---|---:|---:|
| Model loading time (seconds) | {fp16_summary['model_loading_time_seconds']} | {q4_summary['model_loading_time_seconds']} |
| Generation speed (tokens/sec) | {fp16_summary['average_generation_tokens_per_second']} | {q4_summary['average_generation_tokens_per_second']} |
| Peak allocated GPU memory (GB) | {fp16_summary['peak_gpu_memory_allocated_gb']} | {q4_summary['peak_gpu_memory_allocated_gb']} |
| Peak reserved GPU memory (GB) | {fp16_summary['peak_gpu_memory_reserved_gb']} | {q4_summary['peak_gpu_memory_reserved_gb']} |

Generation speed is calculated using total generated tokens divided
by total generation time across the ten prompts.

Memory figures are PyTorch GPU allocation measurements and include
the loaded model during inference.

## Quality evaluation
The same ten prompts were evaluated under both configurations.
The quality_comparison.csv file contains both generated responses
for manual comparison.

The final quality conclusions should be based on the completed
manual assessments, not assumed equivalence.

## Quantization
FP16 represents model weights using 16-bit floating-point values.
4-bit NF4 stores weights using a lower-precision representation,
reducing model memory requirements.

This can improve memory efficiency and allow larger models to fit
on limited hardware, with a possible trade-off in model quality.

## KV cache
The key-value cache stores attention keys and values from previous
tokens during autoregressive generation. It avoids recalculating
them for every newly generated token.

The cache consumes additional memory, increasing with context
length, concurrent requests and model architecture.

## Deployment plan: 20 concurrent users on a 24GB GPU
- Serve a quantized model using an inference engine such as vLLM.
- Use continuous batching to combine active requests.
- Use PagedAttention or equivalent KV-cache management.
- Apply request queues, concurrency limits and rate limits.
- Set context-length and maximum-generation limits.
- Monitor GPU memory, throughput and latency.
- Load-test with realistic traffic before production deployment.
- Scale to additional GPUs if the measured capacity is insufficient.

For the Qwen2.5-1.5B architecture, assuming an FP16 KV cache,
28 layers, 2 KV heads and 128-dimensional KV heads, 20 users
with 2,048 cached tokens each require approximately
{kv_cache_gib:.2f} GiB of KV cache.

This is a theoretical estimate, not a measured concurrent-user test.
The production memory budget must also account for model weights,
runtime overhead, temporary buffers and batching.

## Limitations
- Benchmark conducted on a Colab GPU, not a production 24GB GPU.
- No actual 20-user load test was performed.
- Evaluation used only ten prompts.
- Quality assessment is manual rather than an automated accuracy metric.
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
    f.write(readme)

# Copy updated files to Drive
for filename in os.listdir(OUTPUT_DIR):
    shutil.copy2(
        os.path.join(OUTPUT_DIR, filename),
        os.path.join(DRIVE_DIR, filename)
    )

print("README created and results synchronized to Drive.")
print("KV cache estimate:", round(kv_cache_gib, 2), "GiB")

README created and results synchronized to Drive.
KV cache estimate: 1.09 GiB


In [13]:
import pandas as pd

path = "/content/drive/MyDrive/Tndrr_Task3_Inference/quality_comparison.csv"

quality_df = pd.read_csv(path)

for i, row in quality_df.iterrows():
    print(f"\n{'='*60}")
    print(f"PROMPT {row['Prompt ID']}: {row['Prompt']}")
    print("\n--- FP16 ANSWER ---")
    print(row['FP16 Answer'])
    print("\n--- 4-BIT ANSWER ---")
    print(row['4-bit Answer'])


PROMPT 1: Explain machine learning in simple terms.

--- FP16 ANSWER ---
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms and statistical models to analyze data and make predictions or decisions based on patterns found within the data.

In simpler terms, imagine you have a large dataset containing information about different types of fruits (e.g., apples, bananas, oranges). Machine learning can help us create a model that learns how these fruits behave and classify new fruits into their correct categories based on this learned behavior. This way, we don't need to manually label each fruit; instead, the computer uses its "learning" ability to recognize patterns and categor

--- 4-BIT ANSWER ---
Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance on specific tasks over time without being

In [14]:
import pandas as pd

path = "/content/drive/MyDrive/Tndrr_Task3_Inference/quality_comparison.csv"

df = pd.read_csv(path)

assessments = [
    ("Equivalent", "Both explain machine learning and provide examples."),
    ("Equivalent", "Both correctly explain AI and deep learning."),
    ("Equivalent", "Both provide prime-checking code; complete code is truncated."),
    ("4-bit better", "Provides a more practical and structured notes-app REST API example."),
    ("FP16 better", "FP16 correctly identifies reduced redundancy and easier maintenance; the 4-bit performance claim is questionable."),
    ("Equivalent", "Both accurately explain overfitting and prevention techniques."),
    ("4-bit better", "Explicitly includes IaaS, PaaS and SaaS in the visible response."),
    ("Equivalent", "Both correctly explain binary search on sorted data."),
    ("4-bit better", "Identifies scalability and performance as an additional deployment challenge."),
    ("Equivalent", "Both cover authentication, software updates and encryption.")
]

df["Quality Assessment"] = [a[0] for a in assessments]
df["Observations"] = [a[1] for a in assessments]

df.to_csv(path, index=False)

print("Quality comparison saved!")
print(df[["Prompt ID", "Quality Assessment", "Observations"]].to_string(index=False))

Quality comparison saved!
 Prompt ID Quality Assessment                                                                                                      Observations
         1         Equivalent                                                               Both explain machine learning and provide examples.
         2         Equivalent                                                                      Both correctly explain AI and deep learning.
         3         Equivalent                                                     Both provide prime-checking code; complete code is truncated.
         4       4-bit better                                              Provides a more practical and structured notes-app REST API example.
         5        FP16 better FP16 correctly identifies reduced redundancy and easier maintenance; the 4-bit performance claim is questionable.
         6         Equivalent                                                    Both accurately explain overf

In [15]:
path = "/content/drive/MyDrive/Tndrr_Task3_Inference/README.md"

with open(path, "r", encoding="utf-8") as f:
    print(f.read())


# Task 3: Local LLM Inference and Quantization

## Objective
Compare FP16 and 4-bit quantized inference using the same
model, prompts, generation settings and GPU environment.

## Model and environment
- Model: Qwen/Qwen2.5-1.5B-Instruct
- Runtime: Hugging Face Transformers
- Quantization: bitsandbytes NF4, double quantization enabled
- GPU: Tesla T4
- Evaluation prompts: 10
- Maximum generated tokens: 128
- Sampling: Disabled (greedy decoding)

## Benchmark results

| Metric | FP16 | 4-bit NF4 |
|---|---:|---:|
| Model loading time (seconds) | 82.46 | 14.43 |
| Generation speed (tokens/sec) | 26.51 | 12.83 |
| Peak allocated GPU memory (GB) | 2.89 | 1.092 |
| Peak reserved GPU memory (GB) | 2.934 | 2.918 |

Generation speed is calculated using total generated tokens divided
by total generation time across the ten prompts.

Memory figures are PyTorch GPU allocation measurements and include
the loaded model during inference.

## Quality evaluation
The same ten prompts were evaluated un

In [16]:
from pathlib import Path

readme_path = Path(
    "/content/drive/MyDrive/Tndrr_Task3_Inference/README.md"
)

# Read existing README
readme = readme_path.read_text(encoding="utf-8")

# Updated quality evaluation section
new_section = """## Quality evaluation

The same ten prompts were evaluated under both configurations
using manual comparison of the generated responses.

| Prompt | Assessment |
|---|---|
| 1. Machine learning | Equivalent |
| 2. AI vs deep learning | Equivalent |
| 3. Prime-checking function | Equivalent |
| 4. REST APIs | 4-bit better |
| 5. Database normalization | FP16 better |
| 6. Overfitting | Equivalent |
| 7. Cloud computing | 4-bit better |
| 8. Binary search | Equivalent |
| 9. AI deployment challenges | 4-bit better |
| 10. Cybersecurity | Equivalent |

Overall, six prompts were assessed as equivalent, three favored
4-bit NF4, and one favored FP16.

These are qualitative judgments based on manual inspection,
not an automated quality score. Responses were limited to
128 generated tokens, so some answers were truncated.
The comparison therefore reflects the visible outputs rather
than complete responses.

"""

# Replace the existing section without changing other sections
start = readme.find("## Quality evaluation")
end = readme.find("\n## Quantization", start)

if start == -1 or end == -1:
    raise ValueError("Could not find the expected README section headings.")

updated_readme = readme[:start] + new_section + readme[end:]

# Save updated README
readme_path.write_text(updated_readme, encoding="utf-8")

print("README updated and saved successfully!")
print("\nUpdated quality evaluation section:\n")
print(new_section)

README updated and saved successfully!

Updated quality evaluation section:

## Quality evaluation

The same ten prompts were evaluated under both configurations
using manual comparison of the generated responses.

| Prompt | Assessment |
|---|---|
| 1. Machine learning | Equivalent |
| 2. AI vs deep learning | Equivalent |
| 3. Prime-checking function | Equivalent |
| 4. REST APIs | 4-bit better |
| 5. Database normalization | FP16 better |
| 6. Overfitting | Equivalent |
| 7. Cloud computing | 4-bit better |
| 8. Binary search | Equivalent |
| 9. AI deployment challenges | 4-bit better |
| 10. Cybersecurity | Equivalent |

Overall, six prompts were assessed as equivalent, three favored
4-bit NF4, and one favored FP16.

These are qualitative judgments based on manual inspection,
not an automated quality score. Responses were limited to
128 generated tokens, so some answers were truncated.
The comparison therefore reflects the visible outputs rather
than complete responses.




In [17]:
print(readme_path.read_text(encoding="utf-8"))


# Task 3: Local LLM Inference and Quantization

## Objective
Compare FP16 and 4-bit quantized inference using the same
model, prompts, generation settings and GPU environment.

## Model and environment
- Model: Qwen/Qwen2.5-1.5B-Instruct
- Runtime: Hugging Face Transformers
- Quantization: bitsandbytes NF4, double quantization enabled
- GPU: Tesla T4
- Evaluation prompts: 10
- Maximum generated tokens: 128
- Sampling: Disabled (greedy decoding)

## Benchmark results

| Metric | FP16 | 4-bit NF4 |
|---|---:|---:|
| Model loading time (seconds) | 82.46 | 14.43 |
| Generation speed (tokens/sec) | 26.51 | 12.83 |
| Peak allocated GPU memory (GB) | 2.89 | 1.092 |
| Peak reserved GPU memory (GB) | 2.934 | 2.918 |

Generation speed is calculated using total generated tokens divided
by total generation time across the ten prompts.

Memory figures are PyTorch GPU allocation measurements and include
the loaded model during inference.

## Quality evaluation

The same ten prompts were evaluated u